# APIs Lab

In this lab, you will work with different APIs to build datasets and perform basic analysis.

The goal is to practice how to:

- Make API requests  
- Explore JSON responses  
- Extract relevant information  
- Transform data into pandas DataFrames  
- Generate simple insights  

You will work like a data analyst using APIs data.

# Challenge 1: Pokémon Dataset & Analysis

## Goal

Build and analyze a Pokémon dataset using the API.

## Example (single request)

Below you can see how to get data for ONE Pokémon:

In [ ]:
import requests

url = "https://pokeapi.co/api/v2/pokemon/1"

response = requests.get(url)
data = response.json()

data

## Tasks

1. Using the example above, get data for Pokémon IDs from 1 to 151

2. Extract:
   - name
   - height
   - weight
   - base_experience
   - first type

3. Create a DataFrame

4. Add a new column:
   - weight_kg (divide weight by 10)

5. Answer:
   - Which Pokémon is the heaviest?
   - What is the average weight?
   - How many Pokémon are there per type?

6. Answer:
   - Which type has the highest average weight?

In [27]:
import pandas as pd
import requests

# Base container to collect data
pokemon_list = []

print("Fetching data for Pokémon IDs 1 to 151... Please wait.")

# 1. Fetch data for Pokémon IDs from 1 to 151
for pokemon_id in range(1, 152):
    url = f"https://pokeapi.co/api/v2/pokemon/{pokemon_id}"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()

        # Extract specific properties
        name = data.get("name")
        height = data.get("height")
        weight = data.get("weight")
        base_experience = data.get("base_experience")

        # Extract first type
        types = data.get("types", [])
        first_type = types[0]["type"]["name"] if types else None

        # Append structured dictionary to our list
        pokemon_list.append(
            {
                "name": name,
                "height": height,
                "weight": weight,
                "base_experience": base_experience,
                "first_type": first_type,
            }
        )
    else:
        print(f"Failed to fetch ID {pokemon_id}")

# 2. Convert into a pandas DataFrame
df_pokemon = pd.DataFrame(pokemon_list)

# 3. Add a new column transforming weight to kilograms
df_pokemon["weight_kg"] = df_pokemon["weight"] / 10


# Q1: Which Pokémon is the heaviest?
heaviest_pokemon = df_pokemon.loc[df_pokemon["weight_kg"].idxmax()]

# Q2: What is the average weight?
average_weight_kg = df_pokemon["weight_kg"].mean()

# Q3: How many Pokémon are there per type?
pokemon_per_type = df_pokemon["first_type"].value_counts()

# Q4: Which type has the highest average weight?
type_highest_avg_weight = (
    df_pokemon.groupby("first_type")["weight_kg"].mean().idxmax()
)
highest_avg_weight_val = (
    df_pokemon.groupby("first_type")["weight_kg"].mean().max()
)


# Print Tabular Findings

print("\n" + "=" * 50)
print("             LAB ANALYSIS RESULTS             ")
print("=" * 50)

print(
    f"1. Heaviest Pokémon: {heaviest_pokemon['name'].capitalize()} ({heaviest_pokemon['weight_kg']} kg)"
)
print(f"2. Average weight of Gen 1 Pokémon: {average_weight_kg:.2f} kg")

print("\n3. Pokémon Count Per First Type:")
print(pokemon_per_type)

print(
    f"\n4. Type with the highest average weight: {type_highest_avg_weight.capitalize()} ({highest_avg_weight_val:.2f} kg)"
)




Fetching data for Pokémon IDs 1 to 151... Please wait.

             LAB ANALYSIS RESULTS             
1. Heaviest Pokémon: Snorlax (460.0 kg)
2. Average weight of Gen 1 Pokémon: 45.95 kg

3. Pokémon Count Per First Type:
first_type
water       28
normal      22
poison      14
grass       12
fire        12
bug         12
electric     9
rock         9
ground       8
psychic      8
fighting     7
ghost        3
dragon       3
fairy        2
ice          2
Name: count, dtype: int64

4. Type with the highest average weight: Rock (87.61 kg)


# Challenge 2: Crypto Market Analysis

## Goal

Analyze cryptocurrency prices using real-time data.

## Tasks

1. Get prices for:
   - bitcoin
   - ethereum
   - solana
   - dogecoin

2. Convert the response into a DataFrame (coins as rows)

3. Rename columns properly

4. Add:
   - price_rank (ranking by price)
   - price_category:
        - "high" if > 1000  
        - "medium" if between 100 and 1000  
        - "low" if < 100  

5. Answer:
   - Which coin is the most expensive?
   - Which category has more coins?

6. **Bonus**:
   - Create a column with price difference vs bitcoin

In [26]:
# starter code

import requests
import pandas as pd

url = "https://api.coingecko.com/api/v3/simple/price"

params = {
    # complete this
}


In [28]:
import pandas as pd
import requests


url = "https://api.binance.com/api/v3/ticker/price"

print("Connecting to live exchange servers for real-time market data...")


response = requests.get(url)

if response.status_code == 200:
    raw_data = response.json()

    # Define the specific coins requested by the assignment layout rules
    name_map = {
        "BTCUSDT": "bitcoin",
        "ETHUSDT": "ethereum",
        "SOLUSDT": "solana",
        "DOGEUSDT": "dogecoin",
    }

    coins = []
    prices = []

    # Filter out only the targets specified in the task
    for item in raw_data:
        symbol = item["symbol"]
        if symbol in name_map:
            coins.append(name_map[symbol])
            prices.append(float(item["price"]))

    # Convert values into a clean tabular DataFrame matrix (coins as rows)
    df = pd.DataFrame()
    df["coin"] = coins
    df["price_usd"] = prices

    # Add price_rank column (1 = highest live market value)
    df["price_rank"] = df["price_usd"].rank(ascending=False).astype(int)

    # Filter categories using simple if/elif checks inside a loop container
    categories = []
    for p in df["price_usd"]:
        if p > 1000:
            categories.append("high")
        elif p >= 100:
            categories.append("medium")
        else:
            categories.append("low")

    df["price_category"] = categories

    # Bonus task: Create a column calculating price differences vs bitcoin
    btc_idx = coins.index("bitcoin")
    bitcoin_live_price = prices[btc_idx]

    df["diff_vs_bitcoin"] = df["price_usd"] - bitcoin_live_price


    # OUTPUT REAL-TIME DATAFRAME & ANSWERS
   
    print("\n--- LIVE CRYPTO DATAFRAME ---")
    print(df.to_string(index=False))

    print("\n--- CHALLENGE LAB ANSWERS ---")

    # Answer Question 1: Most expensive coin
    max_idx = df["price_usd"].idxmax()
    print(
        f"1. Most expensive coin right now: {df.loc[max_idx, 'coin'].capitalize()}"
    )

    # Answer Question 2: Largest category count grouping
    counts = df["price_category"].value_counts()
    print(
        f"2. Category with more coins: '{counts.idxmax()}' category (Count: {counts.max()} coins)"
    )

else:
    print(f"Server connection lost. Error Response: {response.status_code}")




Connecting to live exchange servers for real-time market data...

--- LIVE CRYPTO DATAFRAME ---
    coin   price_usd  price_rank price_category  diff_vs_bitcoin
 bitcoin 77534.00000           1           high          0.00000
ethereum  2288.93000           2           high     -75245.07000
dogecoin     0.07517           4            low     -77533.92483
  solana    93.36000           3            low     -77440.64000

--- CHALLENGE LAB ANSWERS ---
1. Most expensive coin right now: Bitcoin
2. Category with more coins: 'high' category (Count: 2 coins)
